In [1]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

# 2. Define the path to your file
# Based on your screenshot, the path is:
file_path = '/content/drive/MyDrive/1. Introduction_to_AI/phishing email detection/cleaned_phishing_emails.csv'

# 3. Load the dataset
try:
    df = pd.read_csv(file_path)
    print(" Data loaded successfully!")
    print("Dataset Shape:", df.shape)
    print("\nFirst 5 rows:")
    display(df.head())
except FileNotFoundError:
    print(" File not found. Please check the exact folder names and file name in your Google Drive.")

Mounted at /content/drive
 Data loaded successfully!
Dataset Shape: (18634, 2)

First 5 rows:


,Email Text,Email Type
0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,the other side of * galicismos * * galicismo *...,Safe Email
2,re : equistar deal tickets are you still avail...,Safe Email
3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [2]:
# 1. Check for missing values
df = df.dropna()

# 2. Remove duplicates to prevent data leakage
df = df.drop_duplicates()

# 3. Identify your text and label columns
# NOTE: You may need to change 'Email Text' and 'Label' to match your exact column names.
# Run print(df.columns) if you get an error here.
text_column = 'Email Text' # Change this if your column is named differently (e.g., 'text', 'body')
label_column = 'Email Type' # Change this if your column is named differently (e.g., 'label', 'class')

# Ensure the columns exist
if text_column not in df.columns or label_column not in df.columns:
    print(f"⚠️ Column names mismatch. Available columns are: {df.columns.tolist()}")
    print("Please update the 'text_column' and 'label_column' variables above.")
else:
    # Keep only the necessary columns
    df = df[[text_column, label_column]]

    # Check class distribution
    print("\nClass Distribution:")
    print(df[label_column].value_counts())

    # Convert labels to binary (0 for Safe, 1 for Phishing)
    # Adjust this based on what your actual labels are (e.g., 'Phishing Email', 'Safe Email')
    df[label_column] = df[label_column].map({'Phishing Email': 1, 'Safe Email': 0})

    # Drop any rows where mapping failed
    df = df.dropna()

    print("\nData ready for Machine Learning:")
    print(df.head())


Class Distribution:
Email Type
Safe Email        10980
Phishing Email     6558
Name: count, dtype: int64

Data ready for Machine Learning:
                                          Email Text  Email Type
0  re : 6 . 1100 , disc : uniformitarianism , re ...           0
1  the other side of * galicismos * * galicismo *...           0
2  re : equistar deal tickets are you still avail...           0
3  \nHello I am your hot lil horny toy.\n    I am...           1
4  software at incredibly low prices ( 86 % lower...           1


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Separate Features (X) and Target (y)
X = df[text_column]
y = df[label_column]

# 2. Split into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Convert text to numerical features using TF-IDF
print("Vectorizing text...")
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# --- Train Model 1: Logistic Regression ---
print("\n--- Training Logistic Regression ---")
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_vec, y_train)
y_pred_lr = lr_model.predict(X_test_vec)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

# --- Train Model 2: Random Forest ---
print("\n--- Training Random Forest ---")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_vec, y_train)
y_pred_rf = rf_model.predict(X_test_vec)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

Vectorizing text...

--- Training Logistic Regression ---
Logistic Regression Accuracy: 0.9783352337514253

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.99      0.98      2196
           1       0.98      0.97      0.97      1312

    accuracy                           0.98      3508
   macro avg       0.98      0.98      0.98      3508
weighted avg       0.98      0.98      0.98      3508


--- Training Random Forest ---
Random Forest Accuracy: 0.9697833523375142

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.97      0.98      2196
           1       0.95      0.97      0.96      1312

    accuracy                           0.97      3508
   macro avg       0.97      0.97      0.97      3508
weighted avg       0.97      0.97      0.97      3508



In [4]:
def predict_phishing(email_text):
    # Transform the input text using the same vectorizer
    email_vec = vectorizer.transform([email_text])

    # Predict using the Random Forest model (or change to lr_model)
    prediction = rf_model.predict(email_vec)[0]
    probability = rf_model.predict_proba(email_vec)[0]

    if prediction == 1:
        print(" ALERT: This email is likely a PHISHING attempt.")
        print(f"Confidence: {probability[1]*100:.2f}%")
    else:
        print(" This email appears to be SAFE.")
        print(f"Confidence: {probability[0]*100:.2f}%")

# Test Case 1: Obvious Phishing
print("Testing Phishing Email:")
phishing_example = "Dear Customer, your account has been suspended. Click this link immediately to verify your identity and update your password: http://secure-bank-update-login.com"
predict_phishing(phishing_example)

print("\n" + "="*50 + "\n")

# Test Case 2: Normal Email
print("Testing Safe Email:")
safe_example = "Hi team, just a reminder that our weekly meeting is moved to 2 PM tomorrow. Please bring your project updates. Thanks!"
predict_phishing(safe_example)

Testing Phishing Email:
 ALERT: This email is likely a PHISHING attempt.
Confidence: 86.88%


Testing Safe Email:
 This email appears to be SAFE.
Confidence: 95.00%


In [12]:
import gradio as gr
import re
import time
import html
import numpy as np

# ==========================================
# 1. PRODUCTION INFERENCE ENGINE WITH ADVANCED XAI
# ==========================================
def analyze_email_production(email_text):
    start_time = time.time()

    # --- 1. Input Validation ---
    if not email_text or not email_text.strip():
        return (
            "<div class='status-card idle'>SYSTEM READY // AWAITING INPUT</div>",
            "", "", "", "", ""
        )

    # Sanitize input to prevent HTML injection
    safe_email_text = html.escape(email_text)

    # --- 2. ML Model Inference ---
    try:
        email_vec = vectorizer.transform([email_text])
        probabilities = rf_model.predict_proba(email_vec)[0]
        safe_conf = float(probabilities[0] * 100)
        phish_conf = float(probabilities[1] * 100)
    except Exception as e:
        return (
            f"<div class='status-card error'>ENGINE ERROR: {str(e)}</div>",
            "", "", "", "", ""
        )

    # --- 3. ADVANCED XAI: Extract Top Features from TF-IDF ---
    feature_names = vectorizer.get_feature_names_out()

    # Get the non-zero features for this specific email
    non_zero_indices = email_vec.nonzero()[1]

    # Get the model's coefficients (weights) for these features
    # Note: Random Forest doesn't have simple coefficients like Logistic Regression,
    # so we use the feature importances of the forest, or fallback to a heuristic.
    if hasattr(rf_model, 'feature_importances_'):
        weights = rf_model.feature_importances_
    else:
        # Fallback if using Logistic Regression
        weights = rf_model.coef_[0] if len(rf_model.coef_.shape) == 1 else rf_model.coef_[0]

    # Calculate the "impact" of each word in the email
    feature_impacts = []
    for idx in non_zero_indices:
        word = feature_names[idx]
        impact = weights[idx]
        # Only keep words with significant impact
        if impact > 0.0001:
            feature_impacts.append((word, impact))

    # Sort by highest impact
    feature_impacts.sort(key=lambda x: x[1], reverse=True)
    top_ml_features = [word for word, impact in feature_impacts[:5]]

    # --- 4. Heuristic Rule Engine ---
    heuristic_risks = []

    # Urgency
    urgency_words = ["urgent", "immediately", "suspended", "action required", "verify", "alert"]
    found_urgency = [w for w in urgency_words if re.search(rf'\b{w}\b', email_text, re.IGNORECASE)]
    if found_urgency:
        heuristic_risks.append(f"Urgency language ({', '.join(found_urgency[:3])})")

    # URLs
    urls = re.findall(r'(https?://[^\s]+)', email_text)
    suspicious_urls = [u for u in urls if not u.startswith("https://") or any(domain in u for domain in ["bit.ly", "tinyurl", "login-", "update-"])]
    if suspicious_urls:
        heuristic_risks.append("Suspicious or insecure URL")

    # PII
    pii_words = ["password", "credit card", "ssn", "social security", "bank account"]
    found_pii = [w for w in pii_words if re.search(rf'\b{w}\b', email_text, re.IGNORECASE)]
    if found_pii:
        heuristic_risks.append(f"Sensitive info request ({', '.join(found_pii[:2])})")

    # --- 5. Ensemble Risk Calculation ---
    heuristic_penalty = len(heuristic_risks) * 10
    final_risk_score = min(100.0, phish_conf + heuristic_penalty)

    # Determine Verdict
    if final_risk_score >= 75:
        verdict_level = "CRITICAL THREAT"
        verdict_color = "#ef4444"
        verdict_bg = "rgba(239, 68, 68, 0.1)"
        icon = "🚨"
        action_msg = "High risk detected. Do not interact with links or attachments."
    elif final_risk_score >= 40:
        verdict_level = "SUSPICIOUS"
        verdict_color = "#f59e0b"
        verdict_bg = "rgba(245, 158, 11, 0.1)"
        icon = "⚠️"
        action_msg = "Proceed with caution. Verify sender identity."
    else:
        verdict_level = "SECURE"
        verdict_color = "#10b981"
        verdict_bg = "rgba(16, 185, 129, 0.1)"
        icon = "✅"
        action_msg = "No immediate threats detected."

    # --- 6. Generate UI Outputs ---

    # A. Verdict Card
    verdict_html = f"""
    <div style='background-color: {verdict_bg}; border: 1px solid {verdict_color}; border-left: 6px solid {verdict_color}; padding: 20px; border-radius: 8px;'>
        <div style='display: flex; justify-content: space-between; align-items: center;'>
            <h2 style='margin: 0; color: #f8fafc; font-size: 1.5em;'>{icon} {verdict_level}</h2>
            <span style='background: {verdict_color}; color: white; padding: 4px 12px; border-radius: 4px; font-weight: bold; font-size: 0.9em;'>{final_risk_score:.1f}% RISK</span>
        </div>
        <p style='margin: 10px 0 0 0; color: #cbd5e1; font-size: 1.1em;'>{action_msg}</p>
    </div>
    """

    # B. Confidence Bar
    breakdown_html = f"""
    <div style='margin-top: 15px;'>
        <div style='display: flex; justify-content: space-between; margin-bottom: 5px; font-size: 0.85em; font-weight: 600; color: #94a3b8;'>
            <span>Safe</span>
            <span>Malicious</span>
        </div>
        <div style='width: 100%; background-color: #1e293b; border-radius: 10px; height: 16px; overflow: hidden; border: 1px solid #334155;'>
            <div style='width: {final_risk_score}%; background: linear-gradient(90deg, {verdict_color} 0%, #ef4444 100%); height: 100%; transition: width 0.5s ease-in-out; box-shadow: 0 0 10px {verdict_color};'></div>
        </div>
    </div>
    """

    # C. Advanced XAI Breakdown (The "Why")
    ml_features_html = "".join([f"<span style='background: rgba(56, 189, 248, 0.2); color: #38bdf8; border: 1px solid #38bdf8; padding: 2px 8px; border-radius: 4px; margin: 2px; display: inline-block; font-family: monospace;'>{word}</span>" for word in top_ml_features])
    heuristic_html = "".join([f"<span style='background: rgba(245, 158, 11, 0.2); color: #f59e0b; border: 1px solid #f59e0b; padding: 2px 8px; border-radius: 4px; margin: 2px; display: inline-block; font-family: monospace;'>{risk}</span>" for risk in heuristic_risks])

    if not top_ml_features:
        ml_features_html = "<span style='color:#64748b;'>No strong ML indicators found.</span>"
    if not heuristic_risks:
        heuristic_html = "<span style='color:#64748b;'>No heuristic triggers detected.</span>"

    detail_html = f"""
    <div style='font-family: monospace; font-size: 0.9em; background: #0f172a; padding: 15px; border-radius: 6px; border: 1px solid #334155;'>
        <div style='margin-bottom: 10px; font-weight: bold; color: #cbd5e1;'>🧠 ML FEATURE IMPORTANCE (Why the AI flagged it):</div>
        <div style='margin-bottom: 15px;'>{ml_features_html}</div>

        <div style='margin-bottom: 10px; font-weight: bold; color: #cbd5e1;'>⚙️ HEURISTIC TRIGGERS (Rule-based):</div>
        <div style='margin-bottom: 15px;'>{heuristic_html}</div>

        <hr style='margin: 10px 0; border-color: #334155;'>
        <div style='color: #94a3b8;'>Raw ML Confidence: {phish_conf:.2f}% | Heuristic Penalty: +{heuristic_penalty}%</div>
    </div>
    """

    # D. Latency & Metadata
    latency_ms = (time.time() - start_time) * 1000
    metadata_html = f"""
    <div style='font-family: monospace; font-size: 0.8em; color: #64748b; text-align: right;'>
        Inference Time: {latency_ms:.2f}ms | Engine: RF-TFIDF-v2 (XAI)
    </div>
    """

    # E. Payload Inspector (Highlighting both ML and Heuristic triggers)
    highlighted_text = safe_email_text

    # Highlight ML features in RED (Highly dangerous)
    for word in top_ml_features:
        # Use regex to match whole words case-insensitively
        pattern = re.compile(rf'\b({re.escape(word)})\b', re.IGNORECASE)
        highlighted_text = pattern.sub(r"<span style='background-color: rgba(239, 68, 68, 0.3); color: #fca5a5; border-bottom: 2px solid #ef4444; padding: 0 2px; font-weight: bold;'>\1</span>", highlighted_text)

    # Highlight Heuristic triggers in YELLOW (Suspicious)
    for word in found_urgency + found_pii:
        pattern = re.compile(rf'\b({re.escape(word)})\b', re.IGNORECASE)
        highlighted_text = pattern.sub(r"<span style='background-color: rgba(245, 158, 11, 0.3); color: #fcd34d; border-bottom: 2px solid #f59e0b; padding: 0 2px; font-weight: bold;'>\1</span>", highlighted_text)

    # Highlight URLs in YELLOW
    for url in urls:
        safe_url = html.escape(url)
        highlighted_text = highlighted_text.replace(safe_url, f"<span style='background-color: rgba(245, 158, 11, 0.3); color: #fcd34d; border-bottom: 2px solid #f59e0b; padding: 0 2px; font-weight: bold;'>{safe_url}</span>")

    inspection_html = f"<div style='white-space: pre-wrap; font-family: monospace; font-size: 0.9em; line-height: 1.6; color: #e2e8f0; background: #0f172a; padding: 15px; border: 1px solid #334155; border-radius: 6px; max-height: 300px; overflow-y: auto;'>{highlighted_text}</div>"

    return verdict_html, breakdown_html, detail_html, metadata_html, inspection_html


# ==========================================
# 2. EXAMPLES
# ==========================================
examples = [
    ["Dear Customer, your account has been suspended due to suspicious activity. Click this link immediately to verify your identity and restore access: http://secure-bank-update-login.com"],
    ["URGENT: You have won a $1000 Amazon gift card! Reply to this email with your credit card details to claim your prize within 24 hours."],
    ["Hi team, just a reminder that our weekly meeting is moved to 2 PM tomorrow. Please bring your project updates. Thanks!"],
    ["Hi John, I've attached the Q3 financial report for your review. Let me know if you have any questions before Thursday's board meeting. Best, Sarah"]
]

# ==========================================
# 3. DARK CYBER THEME CSS
# ==========================================
advanced_css = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;700&display=swap');

/* Global Background - Dark Slate */
.gradio-container {
    font-family: 'Inter', sans-serif !important;
    background-color: #0f172a !important;
    color: #e2e8f0 !important;
    max-width: 1400px !important;
    margin: 0 auto !important;
}

/* Headers */
.app-header {
    border-bottom: 1px solid #334155;
    padding-bottom: 20px;
    margin-bottom: 30px;
    display: flex;
    align-items: center;
    justify-content: space-between;
}
.app-title {
    font-size: 2.2em;
    font-weight: 800;
    color: #f8fafc !important;
    margin: 0;
    text-shadow: 0 0 20px rgba(56, 189, 248, 0.3);
}
.app-subtitle {
    font-size: 1em;
    color: #94a3b8 !important;
    margin: 5px 0 0 0;
    font-family: 'JetBrains Mono', monospace;
}
.badge-prod {
    background: rgba(56, 189, 248, 0.1);
    color: #38bdf8;
    border: 1px solid #38bdf8;
    padding: 4px 12px;
    border-radius: 20px;
    font-size: 0.75em;
    font-weight: 600;
    letter-spacing: 0.5px;
}

/* Panels (Cards) */
.panel-card {
    background: #1e293b !important;
    border: 1px solid #334155 !important;
    border-radius: 12px !important;
    box-shadow: 0 10px 25px -5px rgba(0, 0, 0, 0.5) !important;
    padding: 24px !important;
}

/* Inputs */
textarea {
    background-color: #0f172a !important;
    border: 1px solid #475569 !important;
    border-radius: 8px !important;
    font-family: 'JetBrains Mono', monospace !important;
    font-size: 14px !important;
    color: #38bdf8 !important; /* Cyan terminal text */
    transition: all 0.2s ease;
}
textarea:focus {
    border-color: #38bdf8 !important;
    box-shadow: 0 0 0 3px rgba(56, 189, 248, 0.2) !important;
}

/* Buttons */
#btn-primary {
    background: linear-gradient(135deg, #0ea5e9, #2563eb) !important;
    color: white !important;
    font-weight: 700 !important;
    border-radius: 8px !important;
    border: none !important;
    padding: 12px 24px !important;
    box-shadow: 0 0 15px rgba(14, 165, 233, 0.4) !important;
    text-transform: uppercase;
    letter-spacing: 1px;
    transition: all 0.3s ease !important;
}
#btn-primary:hover {
    box-shadow: 0 0 25px rgba(14, 165, 233, 0.6) !important;
    transform: translateY(-2px);
}
#btn-secondary {
    background: #1e293b !important;
    color: #94a3b8 !important;
    border: 1px solid #475569 !important;
    font-weight: 600 !important;
    border-radius: 8px !important;
}
#btn-secondary:hover {
    background: #334155 !important;
    color: #f8fafc !important;
}

/* Typography Overrides */
h3, .md-header {
    color: #f8fafc !important;
    font-weight: 600 !important;
    font-size: 1.1em !important;
    margin-bottom: 12px !important;
    display: flex;
    align-items: center;
    gap: 8px;
    font-family: 'JetBrains Mono', monospace;
}
label, span, p, div {
    color: #cbd5e1 !important;
}

/* Status Card Idle */
.status-card {
    text-align: center;
    padding: 40px;
    border: 2px dashed #475569;
    border-radius: 12px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.9em;
    color: #64748b !important;
}
.status-card.error {
    border-color: #ef4444;
    color: #ef4444 !important;
    background: rgba(239, 68, 68, 0.1);
}

/* Examples Table */
.gr-examples td {
    background-color: #0f172a !important;
    color: #94a3b8 !important;
    border-bottom: 1px solid #334155 !important;
}
.gr-examples tr:hover td {
    background-color: #1e293b !important;
    color: #38bdf8 !important;
}

/* Hide Gradio Defaults */
.block { border: none !important; box-shadow: none !important; background: transparent !important; }
"""

# ==========================================
# 4. UI LAYOUT
# ==========================================
with gr.Blocks(css=advanced_css, theme=gr.themes.Base()) as demo:

    # Header
    gr.HTML("""
        <div class='app-header'>
            <div>
                <h1 class='app-title'>🛡️ Threat Intelligence Gateway</h1>
                <p class='app-subtitle'>[ ML-POWERED THREAT ANALYSIS & XAI ENGINE ]</p>
            </div>
            <div class='badge-prod'>v2.0 // PRODUCTION</div>
        </div>
    """)

    with gr.Row(equal_height=False):

        # LEFT COLUMN: Input
        with gr.Column(scale=5, elem_classes="panel-card"):
            gr.HTML("<div class='md-header'>📥 INPUT TERMINAL</div>")
            gr.HTML("<p style='font-size:0.9em; color:#94a3b8; margin-bottom:15px;'>Paste raw email content for deep inspection and XAI analysis.</p>")

            email_input = gr.Textbox(
                show_label=False,
                placeholder="Paste email raw text here...",
                lines=14,
            )

            with gr.Row():
                analyze_btn = gr.Button("INITIATE SCAN", elem_id="btn-primary", scale=3)
                clear_btn = gr.Button("RESET", elem_id="btn-secondary", scale=1)

            gr.HTML("<hr style='margin: 25px 0; border-color: #334155;'>")
            gr.HTML("<div class='md-header'>📌 THREAT SAMPLES</div>")
            gr.Examples(
                examples=examples,
                inputs=email_input,
                label=None
            )

        # RIGHT COLUMN: Output
        with gr.Column(scale=7, elem_classes="panel-card"):
            gr.HTML("<div class='md-header'>📡 ANALYSIS OUTPUT</div>")

            # Verdict
            verdict_output = gr.HTML(
                value="<div class='status-card'>SYSTEM READY // AWAITING INPUT</div>"
            )

            # Confidence Bar
            breakdown_output = gr.HTML(value="")

            # Advanced XAI Details
            gr.HTML("<hr style='margin: 20px 0; border-color: #334155;'>")
            gr.HTML("<div class='md-header'>🧠 EXPLAINABLE AI (XAI) BREAKDOWN</div>")
            detail_output = gr.HTML(value="<p style='color:#64748b; font-size:0.9em;'>XAI analysis will appear here.</p>")

            # Metadata
            metadata_output = gr.HTML(value="")

            # Payload Inspector
            gr.HTML("<hr style='margin: 20px 0; border-color: #334155;'>")
            gr.HTML("<div class='md-header'>🔬 PAYLOAD INSPECTOR</div>")
            gr.HTML("<p style='font-size:0.9em; color:#94a3b8; margin-bottom:10px;'><span style='color:#ef4444; font-weight:bold;'>Red</span> = ML Flagged | <span style='color:#f59e0b; font-weight:bold;'>Yellow</span> = Heuristic Flagged</p>")
            inspection_output = gr.HTML(
                value="<div class='status-card'>No payload to inspect.</div>"
            )

    # ==========================================
    # 5. EVENT HANDLERS
    # ==========================================

    analyze_btn.click(
        fn=analyze_email_production,
        inputs=email_input,
        outputs=[verdict_output, breakdown_output, detail_output, metadata_output, inspection_output],
    )

    clear_btn.click(
        fn=lambda: (
            "",
            "<div class='status-card'>SYSTEM READY // AWAITING INPUT</div>",
            "",
            "<p style='color:#64748b; font-size:0.9em;'>XAI analysis will appear here.</p>",
            "",
            "<div class='status-card'>No payload to inspect.</div>"
        ),
        inputs=None,
        outputs=[email_input, verdict_output, breakdown_output, detail_output, metadata_output, inspection_output],
    )

# Launch
demo.launch(share=True, debug=False)

/tmp/ipykernel_439/4035258022.py:348: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=advanced_css, theme=gr.themes.Base()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a410112938ca3c3805.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
